# Behavioral Learning Training 

## **Objective: Train a Temporal Convolutional Network (TCN) for behavioral learning, evaluate performance, and export weights.**

### **Overview**

This notebook implements a **causal Temporal Convolutional Network (TCN)** using PyTorch to learn driving behavior from F1 telemetry sequences.

### **Key Architecture:**
- **Model Type**: Causal TCN (no future information leakage)
- **Input**: T=60 timesteps of telemetry (speed, steering, throttle, brake, distance, etc.)
- **Output**: Next timestep actions (steering, throttle, brake)
- **Training Strategy**: Sliding windows with stride=10 from lap sequences

### **Workflow Sections:**
1. **Setup & Data Loading**: Import libraries, load train/val/test splits, configure hyperparameters
2. **Dataset Preparation**: Create sliding windows with T=60, normalize features, build DataLoaders
3. **TCN Architecture**: Implement causal convolutions, residual blocks, full TCN model
4. **Training Loop**: Train with validation, early stopping, checkpoint best model
5. **Evaluation & Analysis**: Test set metrics, prediction visualization, error analysis
6. **Model Export**: Save weights, metadata, training artifacts: StandardScaler, model architecture in JSON format, input/output feature names.


---

## SECTION 0: Setup & Data Loading

### **📋 Section 0 Overview**

This section prepares the training environment:
- **Import PyTorch and essential libraries** for deep learning
- **Configure device** (GPU/CPU) for optimal performance
- **Set random seeds** for reproducibility (SEED=42)
- **Define hyperparameters** (T=60, stride, batch size, learning rate, epochs)
- **Load train/val/test splits** from CSV files created in N00
- **Setup output directories** for model checkpoints and results

**Output**: Configured environment ready for dataset preparation and model training.

In [13]:
# Core imports
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from datetime import datetime

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Scikit-learn for preprocessing
from sklearn.preprocessing import StandardScaler

# Check PyTorch version
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")

🔥 PyTorch version: 2.1.0+cu118
✅ CUDA available: True
🎮 GPU: NVIDIA GeForce RTX 4060 Laptop GPU


### **⚙️ Configuration & Hyperparameters**

In [14]:
# ========================================
# REPRODUCIBILITY SETTINGS
# ========================================
SEED = 42

def set_seed(seed=42):
    """Set random seeds for reproducibility across numpy, torch, and CUDA."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True  # Enable for Ada Lovelace optimization
    print(f"🎲 Random seed set to {seed} for reproducibility")

set_seed(SEED)

# ========================================
# DEVICE CONFIGURATION
# ========================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Training device: {DEVICE}")

# Enable TF32 for RTX 4060 (Ada Lovelace) - Faster training with minimal accuracy loss
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f"⚡ TF32 enabled for Ada Lovelace acceleration")

# ========================================
# DATA PARAMETERS (from N00_BL_data_split)
# ========================================
T = 60              # Window size in timesteps (6 seconds at 10Hz)
STRIDE = 10         # Sliding window stride for overlap

# ========================================
# MODEL HYPERPARAMETERS (Optimized for RTX 4060 Laptop 8GB VRAM)
# ========================================
BATCH_SIZE = 32     # Reduced from 64 to fit 8GB VRAM comfortably
LEARNING_RATE = 1e-3  # Adam optimizer learning rate
EPOCHS = 50         # Maximum training epochs
PATIENCE = 10       # Early stopping patience

# TCN Architecture parameters (Optimized for memory efficiency)
NUM_CHANNELS = [32, 64, 96, 64]  # Reduced 128->96 to save VRAM, maintain performance
KERNEL_SIZE = 3     # Convolution kernel size
DROPOUT = 0.2       # Dropout rate for regularization

# DataLoader workers (Optimized for Ryzen 9 8945HS - 8 cores/16 threads)
NUM_WORKERS = 6     # Use 6 workers to leave headroom for OS + other processes
PIN_MEMORY = True   # Enable pinned memory for faster GPU transfers

# ========================================
# PATH CONFIGURATION
# ========================================
ROOT = Path.cwd().parents[1]  # Project root
DATA_DIR = ROOT / 'data' / 'processed' / 'BL-train-val-test'

# Input split files
TRAIN_FILE = DATA_DIR / 'splits' / 'train_split.csv'
VAL_FILE = DATA_DIR / 'splits' / 'val_split.csv'
TEST_FILE = DATA_DIR / 'splits' / 'test_split.csv'

# Output directories
MODEL_DIR = DATA_DIR / 'models'
RESULTS_DIR = DATA_DIR / 'results'
MODEL_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

print(f"\n📁 Data directory: {DATA_DIR}")
print(f"💾 Model checkpoint directory: {MODEL_DIR}")
print(f"📊 Results directory: {RESULTS_DIR}")

# ========================================
# HARDWARE SUMMARY
# ========================================
print(f"\n🖥️  Hardware Configuration:")
print(f"   GPU: RTX 4060 Laptop (8GB VRAM)")
print(f"   CPU: Ryzen 9 8945HS (8C/16T)")
print(f"   Batch size: {BATCH_SIZE} (optimized for 8GB VRAM)")
print(f"   DataLoader workers: {NUM_WORKERS}")
print(f"   TCN channels: {NUM_CHANNELS}")

🎲 Random seed set to 42 for reproducibility
🖥️  Training device: cuda
⚡ TF32 enabled for Ada Lovelace acceleration

📁 Data directory: c:\Users\victo\Desktop\Documents\Cuarto Año\Primer Cuatrimestre\F1_AC_Digital_Twin\data\processed\BL-train-val-test
💾 Model checkpoint directory: c:\Users\victo\Desktop\Documents\Cuarto Año\Primer Cuatrimestre\F1_AC_Digital_Twin\data\processed\BL-train-val-test\models
📊 Results directory: c:\Users\victo\Desktop\Documents\Cuarto Año\Primer Cuatrimestre\F1_AC_Digital_Twin\data\processed\BL-train-val-test\results

🖥️  Hardware Configuration:
   GPU: RTX 4060 Laptop (8GB VRAM)
   CPU: Ryzen 9 8945HS (8C/16T)
   Batch size: 32 (optimized for 8GB VRAM)
   DataLoader workers: 6
   TCN channels: [32, 64, 96, 64]


### **🔍 Define Input Features and Target Variables**

In [15]:
# Define input features (telemetry state) and output targets (driver actions)

# INPUT FEATURES: Telemetry sensors and state information
INPUT_FEATURES = [
    'Speed_kmh',      # Current speed
    'Throttle',       # Current throttle input (for context)
    'Brake',          # Current brake input (for context)
    'Steering',       # Current steering input (for context)
    'Distance',       # Position on track
    'RPM',            # Engine RPM
    'Gear'            # Current gear
]

# TARGET OUTPUTS: Driver actions to predict for next timestep
TARGET_FEATURES = [
    'Steering',       # Predicted steering input [-1, 1]
    'Throttle',       # Predicted throttle input [0, 1]
    'Brake'           # Predicted brake input [0, 1]
]

print("🎯 INPUT FEATURES (X):")
for i, feat in enumerate(INPUT_FEATURES, 1):
    print(f"   {i}. {feat}")

print(f"\n📊 Output TARGETS (y):")
for i, feat in enumerate(TARGET_FEATURES, 1):
    print(f"   {i}. {feat}")

print(f"\n📐 Model I/O Shape:")
print(f"   Input: [batch_size, {len(INPUT_FEATURES)}, {T}]")
print(f"   Output: [batch_size, {len(TARGET_FEATURES)}]")

🎯 INPUT FEATURES (X):
   1. Speed_kmh
   2. Throttle
   3. Brake
   4. Steering
   5. Distance
   6. RPM
   7. Gear

📊 Output TARGETS (y):
   1. Steering
   2. Throttle
   3. Brake

📐 Model I/O Shape:
   Input: [batch_size, 7, 60]
   Output: [batch_size, 3]


### **📊 Load Train/Validation/Test Splits**

In [16]:
def load_split_data(file_path):
    """
    Load a train/val/test split CSV file and display summary statistics.
    
    Args:
        file_path (Path): Path to the split CSV file
        
    Returns:
        pd.DataFrame: Loaded telemetry dataframe
    """
    print(f"📂 Loading: {file_path.name}")
    df = pd.read_csv(file_path)
    
    n_laps = df['lap_id'].nunique()
    n_samples = len(df)
    duration_seconds = n_samples * 0.1  # 10Hz sampling
    
    print(f"   ✅ Loaded: {n_samples:,} samples from {n_laps} laps")
    print(f"   ⏱️  Duration: {duration_seconds:.1f}s ({duration_seconds/60:.1f} minutes)")
    print(f"   💾 Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB\n")
    
    return df

In [17]:
# Load all three splits
print("=" * 60)
print("LOADING TRAIN/VALIDATION/TEST SPLITS")
print("=" * 60 + "\n")

df_train = load_split_data(TRAIN_FILE)
df_val = load_split_data(VAL_FILE)
df_test = load_split_data(TEST_FILE)

print("=" * 60)
print("✅ ALL SPLITS LOADED SUCCESSFULLY")
print("=" * 60)

LOADING TRAIN/VALIDATION/TEST SPLITS

📂 Loading: train_split.csv
   ✅ Loaded: 252,588 samples from 307 laps
   ⏱️  Duration: 25258.8s (421.0 minutes)
   💾 Memory: 23.1 MB

📂 Loading: val_split.csv
   ✅ Loaded: 53,350 samples from 65 laps
   ⏱️  Duration: 5335.0s (88.9 minutes)
   💾 Memory: 4.9 MB

📂 Loading: test_split.csv
   ✅ Loaded: 55,062 samples from 67 laps
   ⏱️  Duration: 5506.2s (91.8 minutes)
   💾 Memory: 5.0 MB

✅ ALL SPLITS LOADED SUCCESSFULLY


### **📋 Summary: Configuration Complete**

In [18]:
print("\n" + "=" * 60)
print("✅ SECTION 0 COMPLETE: SETUP & DATA LOADING")
print("=" * 60)

print("\n📋 Configuration Summary:")
print(f"   🎲 Seed: {SEED}")
print(f"   🖥️  Device: {DEVICE}")
print(f"   📏 Window size (T): {T} timesteps")
print(f"   🔄 Stride: {STRIDE}")
print(f"   📦 Batch size: {BATCH_SIZE}")
print(f"   👷 DataLoader workers: {NUM_WORKERS}")
print(f"   📚 Epochs: {EPOCHS}")
print(f"   🎯 Learning rate: {LEARNING_RATE}")
print(f"   ⏸️  Early stopping patience: {PATIENCE}")

print("\n📊 Dataset Statistics:")
print(f"   🏋️  Training samples: {len(df_train):,}")
print(f"   ✅ Validation samples: {len(df_val):,}")
print(f"   🧪 Test samples: {len(df_test):,}")

print("\n🚀 Ready for Section 1: Dataset Preparation with Sliding Windows")
print("=" * 60)


✅ SECTION 0 COMPLETE: SETUP & DATA LOADING

📋 Configuration Summary:
   🎲 Seed: 42
   🖥️  Device: cuda
   📏 Window size (T): 60 timesteps
   🔄 Stride: 10
   📦 Batch size: 32
   👷 DataLoader workers: 6
   📚 Epochs: 50
   🎯 Learning rate: 0.001
   ⏸️  Early stopping patience: 10

📊 Dataset Statistics:
   🏋️  Training samples: 252,588
   ✅ Validation samples: 53,350
   🧪 Test samples: 55,062

🚀 Ready for Section 1: Dataset Preparation with Sliding Windows


In [19]:
# Load all three splits
print("=" * 60)
print("LOADING TRAIN/VALIDATION/TEST SPLITS")
print("=" * 60 + "\n")

df_train = load_split_data(TRAIN_FILE)
df_val = load_split_data(VAL_FILE)
df_test = load_split_data(TEST_FILE)

print("=" * 60)
print("✅ ALL SPLITS LOADED SUCCESSFULLY")
print("=" * 60)

LOADING TRAIN/VALIDATION/TEST SPLITS

📂 Loading: train_split.csv
   ✅ Loaded: 252,588 samples from 307 laps
   ⏱️  Duration: 25258.8s (421.0 minutes)
   💾 Memory: 23.1 MB

📂 Loading: val_split.csv
   ✅ Loaded: 53,350 samples from 65 laps
   ⏱️  Duration: 5335.0s (88.9 minutes)
   💾 Memory: 4.9 MB

📂 Loading: test_split.csv
   ✅ Loaded: 55,062 samples from 67 laps
   ⏱️  Duration: 5506.2s (91.8 minutes)
   💾 Memory: 5.0 MB

✅ ALL SPLITS LOADED SUCCESSFULLY
